# 05. Ranking: датасет кандидатов и опорная линия

Стадия: ranking. Retrieval — iALS со 128 факторами (E05). Ranking-модель будет получать для каждого пользователя 100 кандидатов от retrieval и переупорядочивать их по признакам.

В этом ноутбуке шаг 1: датасет «пользователь, кандидат, метка» без утечки, проверка, что утечки нет, и опорная линия ranking — порядок самого retrieval. Код: `packages/training/src/training/ranking/dataset.py`; данные: `data/processed/ranking_v0/` (собрать: `uv run python -m training.ranking.dataset`).

## Как строится датасет

Метка 1 — кандидат, который пользователь досмотрел, но мы его спрятали. Чтобы retrieval не «знал» спрятанное (утечка), train-пользователей делим на 5 частей, и для каждой части iALS обучается на остальных четырёх. Для своей части эти люди новые, как val-пользователи для итоговой модели:

- `input` — всё, кроме спрятанного: исключается из кандидатов, его `completed` дают вектор пользователя;
- `target` — 20% его `completed` (не меньше одного): это метки 1.

Для val кандидаты дают iALS, обученный на всех train-пользователях: это ровно та ситуация, в которой ranking будет работать.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from training.evaluation import metrics, protocol, stats
from training.interactions import store
from training.ranking import dataset as rd

candidates = pd.read_csv(rd.OUTPUT / "candidates.csv")
targets = pd.read_csv(rd.OUTPUT / "targets.csv")
candidates.head()

In [ ]:
summary = candidates.groupby("group").agg(
    rows=("label", "size"),
    users=("user_id", "nunique"),
    positive_share=("label", "mean"),
)
summary.round(4)

## Проверка, что утечки нет

`recall@100` по частям: какая доля спрятанных ответов попала в 100 кандидатов. Если бы iALS видел спрятанное у train-пользователей, у их частей значение было бы заметно выше, чем у val (в прошлой проверке с утечкой получалось 0,664 против честных 0,577). Здесь они должны быть близки.

In [ ]:
rd.recall_of_candidates(candidates, targets).round(4)

## Насколько полезен балл retrieval

Доля позитивов по позициям в списке кандидатов (корзины по 10 позиций). Если retrieval хороший, доля падает от первых позиций к последним. Ranking имеет смысл, если внутри этого порядка ещё есть что улучшать.

In [ ]:
candidates["rank_bucket"] = (candidates["retrieval_rank"] - 1) // 10 * 10 + 1
by_rank = candidates.groupby(["group", "rank_bucket"])["label"].mean().unstack(0)

ax = by_rank.plot(marker="o", figsize=(7, 3.5))
ax.set_xlabel("позиция у retrieval (начало корзины из 10)")
ax.set_ylabel("доля позитивов")
ax.set_title("Позитивы по позициям кандидатов")
plt.tight_layout()
by_rank.round(3)

## Опорная линия для ranking

Ranking будет переупорядочивать 100 кандидатов. Самое простое — оставить порядок retrieval. Посчитаем на val те же метрики итогового списка, что и раньше (`ndcg@10`, `hit_rate@10`, `recall@10`); спрятанные ответы считаем полностью, включая не попавшие в кандидаты, поэтому числа сравнимы с прошлыми ноутбуками. Любая ranking-модель должна обогнать эту строку.

In [ ]:
table, catalog = store.load_interactions()
cases = protocol.build_cases(table, "val")


def list_metrics(scored: pd.DataFrame, score_column: str, ks=(10, 20)) -> pd.DataFrame:
    """Метрики по каждому val-пользователю для порядка кандидатов по score_column (по убыванию)."""
    rows = {}
    for user_id, group in scored.groupby("user_id"):
        ranked = group.sort_values(score_column, ascending=False)["anime_id"].to_numpy()
        targets_u = cases[user_id][1]
        row = {}
        for k in ks:
            row[f"recall@{k}"] = metrics.recall_at_k(ranked, targets_u, k)
            row[f"hit_rate@{k}"] = metrics.hit_rate_at_k(ranked, targets_u, k)
            row[f"ndcg@{k}"] = metrics.ndcg_at_k(ranked, targets_u, k)
        rows[user_id] = row
    return pd.DataFrame.from_dict(rows, orient="index")


val = candidates[candidates["group"] == "val"]
baseline = list_metrics(val, "retrieval_score")
baseline_summary = pd.DataFrame({
    m: {"среднее": baseline[m].mean(), **dict(zip(["нижняя", "верхняя"], stats.bootstrap_ci(baseline[m].to_numpy())))}
    for m in ["ndcg@10", "hit_rate@10", "recall@10"]
}).T
baseline_summary.round(4)

Вывод по шагу 1:

- Датасет: 1386 train-пользователей (по 100 кандидатов, всего 138 600 строк) и 173 val-пользователя (17 300 строк); позитивов около 21%.
- Утечки нет: `recall@100` у пяти train-частей 0,617–0,656, у val 0,647. Если бы iALS видел спрятанное, у train было бы заметно выше.
- Значение `recall@100` у val совпадает с результатом E05 (0,6465), то есть кандидаты построены той же моделью, что и в сравнении retrieval.
- Опорная линия ranking — порядок retrieval; её метрики в таблице выше. Следующий шаг — признаки для каждой пары «пользователь, кандидат» и первая ranking-модель.